## Demonstração do Controller de Transação

Este notebook demonstra o funcionamento do controller de transações da API, cobrindo as seguintes operações:

- Criar transação (`POST /transactions/create`)
- Buscar todas transações (`GET /transactions/all`)
- Buscar transações deletadas (`GET /transactions/deleted`)
- Buscar transações por usuário (`GET /transactions/{user_id}/user`)
- Buscar transação por ID (`GET /transactions/{id}`)
- Atualizar transação (`PATCH /transactions/{id}/update`)
- Deletar (soft delete) transação (`DELETE /transactions/{id}/delete`)
- Restaurar transação (`POST /transactions/{id}/restore`)
- Deletar permanentemente (`DELETE /transactions/{id}/force-delete`)


## Setup do Teste com FastAPI e TestClient

Import e configuração do FastAPI com o router `transactions`.

In [1]:
from fastapi.testclient import TestClient
from fastapi import FastAPI

from user.controller import users_router
from transactions.controller import transactions_router
from category_types.controller import category_types_router
from categories.controller import categories_router

app = FastAPI()
app.include_router(users_router)
app.include_router(category_types_router)
app.include_router(categories_router)
app.include_router(transactions_router)

client = TestClient(app)

## Criar Usuário de Teste

**Endpoint:** `POST /users/create`  
**Descrição:** Cria um novo usuário com os dados fornecidos no payload.

In [2]:
user_payload = {
    "first_name": "Phill",
    "last_name": "Wenneck",
    "cpf": "235.364.190-38",
    "email": "phill@wolfpack.com",
    "password": "VegasBaby123!",
    "manual_balance": 700
}

response = client.post("/users/create", json=user_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_user = response.json()
user_id = created_user["id"]

2025-06-16 21:37:12,546 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2025-06-16 21:37:12,549 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-06-16 21:37:12,552 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2025-06-16 21:37:12,553 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-06-16 21:37:12,554 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2025-06-16 21:37:12,555 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-06-16 21:37:12,557 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:37:12,561 INFO sqlalchemy.engine.Engine INSERT INTO users (id, first_name, last_name, email, cpf, password, manual_balance, created_at, updated_at, deleted_at) VALUES (%(id)s, %(first_name)s, %(last_name)s, %(email)s, %(cpf)s, %(password)s, %(manual_balance)s, %(created_at)s, %(updated_at)s, %(deleted_at)s)
2025-06-16 21:37:12,562 INFO sqlalchemy.engine.Engine [generated in 0.00116s] {'id': '06850b8b88fd705f8000c84f14ba668e', 'first_name': 'Phill', 'last_name': 'Wenneck', 'email'

## Criar Tipo de Categoria de Teste

**Endpoint:** `POST /category_types/create`  
**Descrição:** Cria um novo tipo categoria com os dados fornecidos no payload.

In [3]:
category_type_payload = {
    "name": "Gastos",
    "is_positive": False,
}

response = client.post("/category_types/create", json=category_type_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_category_type = response.json()
category_type_id = created_category_type["id"]

2025-06-16 21:37:34,627 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:37:34,647 INFO sqlalchemy.engine.Engine INSERT INTO category_types (id, name, is_positive) VALUES (%(id)s, %(name)s, %(is_positive)s)
2025-06-16 21:37:34,648 INFO sqlalchemy.engine.Engine [generated in 0.00122s] {'id': '06850b8cea5b77c98000029c62826849', 'name': 'Gastos', 'is_positive': 0}
2025-06-16 21:37:34,663 INFO sqlalchemy.engine.Engine COMMIT
2025-06-16 21:37:34,670 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:37:34,672 INFO sqlalchemy.engine.Engine SELECT category_types.id, category_types.name, category_types.is_positive 
FROM category_types 
WHERE category_types.id = %(pk_1)s
2025-06-16 21:37:34,673 INFO sqlalchemy.engine.Engine [generated in 0.00081s] {'pk_1': '06850b8cea5b77c98000029c62826849'}
2025-06-16 21:37:34,675 INFO sqlalchemy.engine.Engine ROLLBACK
Status: 201
Resposta: {'id': '06850b8c-ea5b-77c9-8000-029c62826849', 'name': 'Gastos', 'is_positive': False}


## Criar Categoria de Teste

**Endpoint:** `POST /category/create`  
**Descrição:** Cria uma nova categoria com os dados fornecidos no payload.

In [4]:
category_payload = {
    "user_id": user_id,
    "category_type_id": category_type_id,
    "name": "Despedida de Solteiro do Doug",
    "description": "Categoria de Gastos da despedida de Solteiro do Doug"
}

response = client.post("/categories/create", json=category_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_category = response.json()
category_id = created_category["id"]

2025-06-16 21:37:40,023 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:37:40,029 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.first_name AS users_first_name, users.last_name AS users_last_name, users.email AS users_email, users.cpf AS users_cpf, users.password AS users_password, users.manual_balance AS users_manual_balance, users.created_at AS users_created_at, users.updated_at AS users_updated_at, users.deleted_at AS users_deleted_at 
FROM users 
WHERE users.id = %(id_1)s AND users.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-16 21:37:40,030 INFO sqlalchemy.engine.Engine [generated in 0.00119s] {'id_1': '06850b8b88fd705f8000c84f14ba668e', 'param_1': 1}
2025-06-16 21:37:40,035 INFO sqlalchemy.engine.Engine SELECT category_types.id AS category_types_id, category_types.name AS category_types_name, category_types.is_positive AS category_types_is_positive 
FROM category_types 
WHERE category_types.id = %(id_1)s 
 LIMIT %(param_1)s
2025-06-16 21:37:40,037

## Criar Transação de Teste

**Endpoint:** `POST /transactions/create`  
**Descrição:** Cria uma nova transação com os dados fornecidos no payload.

In [8]:
transaction_payload = {
    "user_id": user_id,
    "category_id":category_id,
    "name": "Cobertura Caesar Palace",
    "description": "Porque Las Vegas não é lugar pra economizar",
    "amount": 3500.00,
    "date": "2025-06-13",
    "is_recurring": False
}

response = client.post("/transactions/create", json=transaction_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_transaction = response.json()
transaction_id = created_transaction["id"]

2025-06-16 21:38:43,417 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:38:43,421 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.first_name AS users_first_name, users.last_name AS users_last_name, users.email AS users_email, users.cpf AS users_cpf, users.password AS users_password, users.manual_balance AS users_manual_balance, users.created_at AS users_created_at, users.updated_at AS users_updated_at, users.deleted_at AS users_deleted_at 
FROM users 
WHERE users.id = %(id_1)s AND users.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-16 21:38:43,422 INFO sqlalchemy.engine.Engine [cached since 63.39s ago] {'id_1': '06850b8b88fd705f8000c84f14ba668e', 'param_1': 1}
2025-06-16 21:38:43,427 INFO sqlalchemy.engine.Engine SELECT categories.id AS categories_id, categories.user_id AS categories_user_id, categories.category_type_id AS categories_category_type_id, categories.name AS categories_name, categories.description AS categories_description, categories.created_

## Buscar Todas as Transações Ativas

**Endpoint:** `GET /transactions/all`  
**Descrição:** Retorna todas as transações que estão ativos no sistema.

In [9]:
response = client.get("/transactions/all")
print("Status:", response.status_code)
for user in response.json():
    print(user)

2025-06-16 21:38:54,093 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:38:54,096 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.deleted_at IS NULL
2025-06-16 21:38:54,097 INFO sqlalchemy.engine.Engine [cached since 53.89s ago] {}
2025-06-16 21:38:54,10

## Buscar Transações Desativadas

** Endpoint: ** `GET /transactions/deleted`
** Descrição: ** Retorna todas as transações que estão desativadas no sistema.

In [10]:
response = client.get("/transactions/deleted")
print("Status:", response.status_code)
print("Transações desativadas:", response.json())

2025-06-16 21:38:57,259 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:38:57,266 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.deleted_at IS NOT NULL
2025-06-16 21:38:57,267 INFO sqlalchemy.engine.Engine [generated in 0.00093s] {}
2025-06-16 21:38:57,

## Buscar Transações por Usuário
** Endpoint: ** `GET /transactions/{user_id}/user`
** Descrição: ** Retorna as transações identificadas pelo ID do usuário.


In [11]:
response = client.get(f"/transactions/{user_id}/user")
print("Status:", response.status_code)
print("Transações do usuário:", response.json())

2025-06-16 21:39:28,982 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:39:28,989 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.user_id = %(user_id_1)s AND transactions.deleted_at IS NULL
2025-06-16 21:39:28,991 INFO sqlalchemy.engine.Engine [generated

## Buscar Transação por ID

**Endpoint:** `GET /transactions/{id}`  
**Descrição:** Retorna os dados de um Transação específica, identificada pelo ID.

In [12]:
response = client.get(f"/transactions/{user_id}")
print("Status:", response.status_code)
print("Transação:", response.json())

2025-06-16 21:39:35,104 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:39:35,127 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.id = %(id_1)s AND transactions.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-16 21:39:35,128 INFO sqlalchemy.engine.Engine 

## Atualizar Dados do Transação

**Endpoint:** `PATCH /transactions/{id}/update`  
**Descrição:** Atualiza os campos fornecidos na Transação (ex: saldo manual).

In [13]:
update_payload = {
    "amount": 5265.75,
    "date": "2025-10-20"
}

response = client.patch(f"/transactions/{transaction_id}/update", json=update_payload)
print("Status:", response.status_code)
print("Atualizado:", response.json())

2025-06-16 21:40:47,023 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:40:47,027 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.id = %(id_1)s AND transactions.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-16 21:40:47,028 INFO sqlalchemy.engine.Engine 

## Desativar Transação (Soft Delete)

**Endpoint:** `DELETE /transactions/{id}/delete`  
**Descrição:** Marca a Transação como desativada, sem removê-lo do banco de dados.

In [17]:
response = client.delete(f"/transactions/{transaction_id}/delete")
print("Status:", response.status_code)
print("Transação deletado:", response.json())

2025-06-16 21:41:27,518 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:41:27,534 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.id = %(id_1)s AND transactions.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-16 21:41:27,534 INFO sqlalchemy.engine.Engine 

## Restaurar Transação Desativada

**Endpoint:** `POST /transactions/{id}/restore`  
**Descrição:** Restaura uma Transação que foi previamente desativada.

In [15]:
response = client.post(f"/transactions/{transaction_id}/restore")
print("Status:", response.status_code)
print("Restaurado:", response.json())

2025-06-16 21:41:07,414 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:41:07,420 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.id = %(id_1)s AND transactions.deleted_at IS NOT NULL 
 LIMIT %(param_1)s
2025-06-16 21:41:07,420 INFO sqlalchemy.engine.Eng

## Deletar Transação Permanentemente

**Endpoint:** `DELETE /transactions/{id}/force-delete`  
**Descrição:** Deleta permanentemente a Transação do banco de dados (hard delete). A Transação precisa estar desativada antes.

In [20]:
response = client.delete(f"/transactions/{transaction_id}/force-delete")
print("Status:", response.status_code)
print("Forçando a Deleção no banco de dados:", response.status_code)

2025-06-16 21:42:03,086 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-16 21:42:03,103 INFO sqlalchemy.engine.Engine SELECT transactions.id AS transactions_id, transactions.user_id AS transactions_user_id, transactions.account_id AS transactions_account_id, transactions.category_id AS transactions_category_id, transactions.name AS transactions_name, transactions.description AS transactions_description, transactions.amount AS transactions_amount, transactions.date AS transactions_date, transactions.is_recurring AS transactions_is_recurring, transactions.recurrence_interval AS transactions_recurrence_interval, transactions.next_due_date AS transactions_next_due_date, transactions.created_at AS transactions_created_at, transactions.updated_at AS transactions_updated_at, transactions.deleted_at AS transactions_deleted_at 
FROM transactions 
WHERE transactions.id = %(id_1)s AND transactions.deleted_at IS NOT NULL 
 LIMIT %(param_1)s
2025-06-16 21:42:03,104 INFO sqlalchemy.engine.Eng